In [2]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

/Users/lennartredlich/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/lennartredlich/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "files"

with open(DATA_PATH / "linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)

jobs = []
for person_id, cv in enumerate(cvs):
    for job in cv:
        if job["status"] == "ACTIVE":
            jobs.append({**job, "person_id": person_id})

df_active = pd.DataFrame(jobs)
df_active.head()

,organization,linkedin,position,startDate,endDate,status,department,seniority,person_id
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,None,ACTIVE,Other,Management,0
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management,0
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,None,ACTIVE,Other,Professional,0
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,None,ACTIVE,Other,Management,0
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management,0


In [4]:
df_active["seniority"].value_counts()

seniority
Professional    216
Management      192
Lead            125
Senior           44
Director         34
Junior           12
Name: count, dtype: int64

In [5]:
sen_labels = [
    "Professional",
    "Management",
    "Lead",
    "Senior",
    "Director",
    "Junior"
]

sen_label_embeddings = model.encode(sen_labels)

In [11]:
job_embeddings = model.encode(df_active["position"].astype(str).tolist())

sim = cosine_similarity(job_embeddings, sen_label_embeddings)
best_idx = sim.argmax(axis=1)

df_active["seniority_pred_embedding"] = [sen_labels[i] for i in best_idx]
df_active[["position", "seniority", "seniority_pred_embedding"]].head(5)

,position,seniority,seniority_pred_embedding
0,Prokurist,Management,Professional
1,CFO,Management,Management
2,Betriebswirtin,Professional,Lead
3,Prokuristin,Management,Director
4,CFO,Management,Management


In [7]:
sen_embedding_accuracy = (df_active["seniority"] == df_active["seniority_pred_embedding"]).mean()
sen_embedding_accuracy

np.float64(0.36918138041733545)

In [8]:
df_active.loc[
    df_active["seniority"] != df_active["seniority_pred_embedding"],
    ["position", "seniority", "seniority_pred_embedding"]
].head(20)

,position,seniority,seniority_pred_embedding
0,Prokurist,Management,Professional
2,Betriebswirtin,Professional,Lead
3,Prokuristin,Management,Director
5,Solutions Architect,Professional,Management
6,Medizintechnik Beratung,Professional,Management
8,Gerente comercial,Lead,Director
9,Administrador Unico,Professional,Management
10,"APL-ansvarig, samordning",Lead,Senior
11,Kaufmännischer Leiter,Lead,Junior
12,Lab-Supervisor,Lead,Management


In [9]:
pd.crosstab(
    df_active["seniority"],
    df_active["seniority_pred_embedding"],
    rownames=["truth"],
    colnames=["pred"],
    normalize="index"
).round(3)

pred,Director,Junior,Lead,Management,Professional,Senior
truth,,,,,,
Director,0.941,0.000,0.000,0.029,0.029,0.000
Junior,0.167,0.250,0.083,0.083,0.083,0.333
Lead,0.064,0.024,0.104,0.640,0.128,0.040
Management,0.240,0.047,0.026,0.578,0.068,0.042
Professional,0.079,0.088,0.060,0.551,0.190,0.032
Senior,0.023,0.023,0.000,0.182,0.091,0.682


Misclassifications predominantly occur between adjacent seniority levels, particularly between Lead and Management.
This reflects semantic overlap rather than random prediction errors and indicates that the embedding-based model captures hierarchical structure implicitly.

In my Opinion Lead is a little bit to be confused with Management, where exactly is the Difference? 

Each ACTIVE position is treated as an independent prediction instance.
Profiles with multiple concurrent roles are therefore represented by multiple samples.
This design aligns with the intended use case of job-level matching rather than person-level career profiling.

In [10]:
order = ["Junior", "Professional", "Senior", "Lead", "Management", "Director"]
rank = {k: i for i, k in enumerate(order)}

df_active["truth_rank"] = df_active["seniority"].map(rank)
df_active["pred_rank"]  = df_active["seniority_pred_embedding"].map(rank)

# absolute distance in levels
df_active["level_diff"] = (df_active["truth_rank"] - df_active["pred_rank"]).abs()

acc_exact = (df_active["level_diff"] == 0).mean()
acc_pm1   = (df_active["level_diff"] <= 1).mean()
acc_pm2   = (df_active["level_diff"] <= 2).mean()

print("Exact accuracy:", round(acc_exact, 3))
print("Accuracy within ±1 level:", round(acc_pm1, 3))
print("Accuracy within ±2 levels:", round(acc_pm2, 3))

Exact accuracy: 0.369
Accuracy within ±1 level: 0.639
Accuracy within ±2 levels: 0.732
